# 프로젝트: KoChatGPT 업그레이드 하기


In [2]:
import os
import sys
import random
from copy import deepcopy
import json

import torch
import torch.nn as nn
import evaluate
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType, PeftModel

from transformers import (
    PreTrainedTokenizerFast,
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling
)


print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.10.0


In [3]:
CFG = {
    "MODEL_NAME": "skt/ko-gpt-trinity-1.2B-v0.5",
    "KOCHATGPT_PATH": "/Users/jamesyang/Projects/KoChatGPT/data_kochatgpt",
    "MODEL_OUTPUT_PATH": "models/SFT",

    # 학습 도중 체크포인트들이 저장될 위치
    "TRAIN_OUTPUT_PATH": "kochatgpt-trinity-sft",
    "PROMPT_TEMPLATE": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):",
    "LORA_RANK": 8,

    "RM_MODEL_OUTPUT_PATH": "models/RM",
    "PPO_MODEL_OUTPUT_PATH": "models/PPO",

    "DEVICE": torch.device("cuda" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")
}


In [5]:
HOME = os.path.expanduser("~")
BASE_PATH = os.path.join(HOME, "Projects/content")
GPT_PATH =  f"{BASE_PATH}/chatgpt"

if BASE_PATH not in sys.path:
    sys.path.append(BASE_PATH)

from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer
from chatgpt.dataset import RewardDataset
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer


# [Phase 1] Foundation Model 교체 및 LoRA 설정 (SFT 모델 로드)

- 1단계: 원본 모델 가중치 (12억 개 파라미터) ➔ Freeze
  - Trinity 1.2B 모델 본체의 모든 파라미터는 얼음(Frozen) 상태가 됩니다.
  - **미분(Gradient)**을 계산하지 않고, 값을 바꾸지도 않습니다. 학습 시 메모리를 아주 적게 차지하게 됩니다.
- 2단계: LoRA용 얇은 레이어 (약 1,100만 개 파라미터) ➔ Unfreeze (학습 가능)
  - 원본 모델 옆에 아주 얇은 '어댑터(Adapter)' 레이어를 새로 끼워 넣습니다. 이 부분만 학습 가능하도록 열어둡니다.
  - 우리가 준비한  SFT_dataset 의 지식은 오직 이 아주 작은 레이어에만 기록됩니다.
- 3단계: 합치기 (Inference)
  - 생성할 때는 얼어있는 본체와 학습된 어댑터가 힘을 합쳐 답변을 내놓습니다.

In [5]:
# [셀 3] Foundation Model 로드 및 PEFT(LoRA) 적용

# 토크나이저 로드 (special tokens 추가)
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    CFG["MODEL_NAME"], 
    bos_token='</s>', 
    eos_token='</s>', 
    unk_token='<unk>', 
    pad_token='<pad>', 
    mask_token='<mask>',
    padding_side="right",      # 학습을 위해 우측 패딩 설정
    model_max_length=512       # 모델이 한 번에 처리할 최대 길이 설정
)

# 모델 로드 (Mac의 경우 device_map="auto" 로 MPS에 적절히 할당되거나 GPU 메모리 최적화를 위해 fp16 사용)
model = AutoModelForCausalLM.from_pretrained(
    CFG["MODEL_NAME"],
    dtype=torch.float16,  # 메모리 절약을 위한 Half Precision
    device_map="auto"
)

# LoRA(Low-Rank Adaptation) 설정
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=CFG["LORA_RANK"],               # Rank 크기 (줄일수록 학습 파라미터가 적어짐)
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["c_attn"] # GPT-2 계열의 Attention 레이어 Linear Projection 타겟
)

# 모델에 LoRA 어댑터 부착
model = get_peft_model(model, peft_config)

# 학습할 파라미터 수 확인 (전체 파라미터 대비 1% 미만으로 OOM 방지)
model.print_trainable_parameters()


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 1,474,560 || all params: 1,164,030,720 || trainable%: 0.1267


/Users/jamesyang/.pyenv/versions/3.12.2/envs/aipel/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
import json

# 1. 파일 로드
data_path = f"{CFG['KOCHATGPT_PATH']}/kochatgpt_1_SFT.jsonl"
dataset = load_dataset('json', data_files=data_path)

def preprocess_function(examples, prompt_template):
    sources = [prompt_template.format(prompt=p) for p in examples['prompt']]
    targets = [f"\n{c}{tokenizer.eos_token}" for c in examples['completion']]
    full_sentences = [s + t for s, t in zip(sources, targets)]
    
    model_inputs = tokenizer(full_sentences, max_length=512, truncation=True, padding="max_length")
    
    labels = [list(ids) for ids in model_inputs["input_ids"]]
    
    for i, source in enumerate(sources):
        source_len = len(tokenizer(source, truncation=True)["input_ids"])
        # 질문 영역은 -100으로 채워 손실 계산에서 제외
        labels[i] = [-100] * source_len + labels[i][source_len:]
        # [-100, -100, -100, -100, -100, -100, 405, 607, 708, 2, 0, 0, 0, ...]
    
    model_inputs["labels"] = labels
    return model_inputs


tokenized_datasets = dataset.map(
    preprocess_function, 
    batched=True,
    fn_kwargs={"prompt_template": CFG["PROMPT_TEMPLATE"]}, 
    remove_columns=dataset['train'].column_names
)
# 4. 데이터 스플릿
split_dataset = tokenized_datasets['train'].train_test_split(test_size=0.1)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']
print(f"✅ 데이터 준비 및 스플릿 완료! (학습용 샘플: {len(train_dataset)})")

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

✅ 데이터 준비 및 스플릿 완료! (학습용 샘플: 10800)


In [ ]:
# [검증] 첫 번째 데이터를 디코딩해서 확인해보기
sample = train_dataset[0]
# 1. 원본 문장(input_ids) 확인
print("--- [1. 실제 입력 문장] ---")
print(tokenizer.decode(sample['input_ids'], skip_special_tokens=True)[:200] + "...")


# 2. 학습 대상(labels) 확인 (마스킹 확인)
print("\n--- [2. 모델이 학습하는 부분 (Labels)] ---")
# labels에서 -100이 아닌 부분만 추출해서 디코딩
labels = sample['labels']
actual_labels = [token if token != -100 else tokenizer.pad_token_id for token in labels]
decoded_labels = tokenizer.decode(actual_labels, skip_special_tokens=True)
print(f"가려진 부분(처음 10개): {labels[:10]}") # -100이 주르륵 나와야 함
print(f"실제 학습되는 텍스트: {decoded_labels.strip()[:150]}...")

--- [1. 실제 입력 문장] ---
### Instruction(명령어):
서울역사박물관은 뭐하는 장소야

### Response(응답):
'서울역사박물관은 서울의 고유한 역사와 문화, 생활습관 등을 전시하고 보존하는 박물관입니다. 주요 전시물로는 조선시대부터 근현대까지의 서울의 역사를 소개하는 전시물, 서울의 생활문화와 전통재래시장 등을 소개하는 전시물, 서울의 건축물과 도시계획 등을 소개...

--- [2. 모델이 학습하는 부분 (Labels)] ---
가려진 부분(처음 10개): [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
실제 학습되는 텍스트: :
'서울역사박물관은 서울의 고유한 역사와 문화, 생활습관 등을 전시하고 보존하는 박물관입니다. 주요 전시물로는 조선시대부터 근현대까지의 서울의 역사를 소개하는 전시물, 서울의 생활문화와 전통재래시장 등을 소개하는 전시물, 서울의 건축물과 도시계획 등을 소개하는 전시물...


In [19]:
# [셀 5 맨 위에 추가]
import os
import psutil
def show_memory_info():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"현재 메모리 사용량: {mem_info.rss / 1024 / 1024:.2f} MB")

In [20]:
# [셀 5] Trainer 설정 및 SFT 학습 실행
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

# 1. 데이터 콜레이터 (자동으로 패딩을 채워줌)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 2. 학습 파라미터 설정 (Mac M4 / 1.2B 모델 최적화)
training_args = TrainingArguments(
    output_dir=CFG["TRAIN_OUTPUT_PATH"],
    per_device_train_batch_size=1,        # 메모리 부족 시 1로 조절
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,        # 배치 사이즈를 키우는 효과 (2 * 8 = 16)
    num_train_epochs=1,                   # 실습을 위해 우선 1에폭만 진행 (전략적 시간 단축)
    learning_rate=2e-4,                   # LoRA 전용 최적 학습률
    # 🚀 [Mac M4 핵심 설정] 🚀
    fp16=False,                           # MPS 충돌 방지를 위해 끕니다
    bf16=True,                            # M4 가속기를 백분 활용하는 강력 추천 옵션!              # 메모리 절감 및 학습 속도 향상
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    weight_decay=0.01,
    lr_scheduler_type="cosine",           # 학습률을 부드럽게 감소시켜 안정성 향상
    load_best_model_at_end=True,
    report_to="none"                      # 외부 로그(WandB 등) 끄기
)

# 3. Trainer 초기화
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

# 4. SFT 학습 시작 🚀
print("🚀 [SFT Phase] Trinity 1.2B + LoRA 학습을 시작합니다...")
show_memory_info() # 메모리 상태 확인 함수가 있다면 사용 (선택 사항)

trainer.train()

# 5. 학습된 모델(LoRA 어댑터) 저장
model.save_pretrained(CFG["MODEL_OUTPUT_PATH"])
tokenizer.save_pretrained(CFG["MODEL_OUTPUT_PATH"])
print("✅ SFT 학습 완료 및 모델 저장 시스템 성공!")


🚀 [SFT Phase] Trinity 1.2B + LoRA 학습을 시작합니다...
현재 메모리 사용량: 345.30 MB


/Users/jamesyang/.pyenv/versions/3.12.2/envs/aipel/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
50,2.262607,2.042535
100,2.047478,1.944656
150,2.003043,1.903943
200,2.110839,1.882491
250,2.034708,1.869337
300,2.134847,1.859140
350,1.953019,1.857940
400,2.065107,1.842595
450,2.017776,1.836266
500,1.916939,1.829840


/Users/jamesyang/.pyenv/versions/3.12.2/envs/aipel/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/jamesyang/.pyenv/versions/3.12.2/envs/aipel/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/jamesyang/.pyenv/versions/3.12.2/envs/aipel/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/jamesyang/.pyenv/versions/3.12.2/envs/aipel/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't 

✅ SFT 학습 완료 및 모델 저장 시스템 성공!


여기서 PEFT는 Parameter-Efficient Fine-Tuning의 약자로, **"적은 수의 파라미터만 사용해서 효율적으로 미세 조정한다"**는 뜻입니다.

왜 이걸 쓰는지, 어떤 역할을 하는지 3 줄 요약해 드릴게요.

- "전부를 가르치지 않습니다" (얼리기)
원래 Trinity 1.2B 같은 거대 모델을 처음부터 끝까지 다 가르치려면 메모리가 수십 GB가 필요하고 시간도 며칠씩 걸립니다. PeftModel은 원래 모델의 거대한 가중치들은 절대 변하지 않게 꽁꽁 얼려버리고(Freeze), 그 옆에 아주 작은 **'지식 칩(LoRA 어댑터)'**만 붙입니다.

- "지식 칩(Adapter)의 집" (껍데기)
PeftModel은 **[원본 거대 모델] + [우리가 새로 학습시킨 작은 지식 조각]**을 하나로 묶어주는 포장지 같은 역할을 합니다. 우리가 model.generate()라고 명령을 내리면, PeftModel이 알아서 원본 모델의 지식과 우리가 새로 가르친 지식을 잘 버무려서 정답을 내놓게 됩니다.

- "용량 다이어트" (실무적 이유)
전체 모델을 학습하면 3GB짜리 모델 파일을 새로 저장해야 하지만, PeftModel을 쓰면 우리가 학습한 LoRA 어댑터만 따로 저장할 수 있습니다. 그래서 결과물 파일 용량이 수십 MB 수준으로 확 줄어듭니다.

In [ ]:
# 모델 로드 및 LoRA 어댑터 결합
print("🚀 원본 모델(Base Model) 로드 중...")
base_model = AutoModelForCausalLM.from_pretrained(
    CFG["MODEL_NAME"],      # "skt/ko-gpt-trinity-1.2B-v0.5"
    torch_dtype=torch.float16,
    device_map="auto"
)

print("🚀 LoRA 어댑터 부착 중...")
model = PeftModel.from_pretrained(base_model, CFG["MODEL_OUTPUT_PATH"])
model.eval() # 추론 모드로 전환


# 토크나이저 로드 (학습 시 저장한 경로에서 가져옴)
tokenizer = PreTrainedTokenizerFast.from_pretrained(CFG["MODEL_OUTPUT_PATH"])

# 평가용 메트릭 로드
# 어원: Recall-Oriented Understudy for Gisting Evaluation
# 설명: 모델이 낸 답변이 실제 정답 데이터에 포함된 단어들을 얼마나 많이 '포함(Recall)'하고 있는지 측정합니다.
rouge = evaluate.load("rouge")

# 어원: BiLingual Evaluation Understudy
# 설명: 모델이 생성한 문장의 단어 조합이 정답 문장의 단어 조합과 얼마나 '일치(Precision)'하는지 측정합니다.
bleu = evaluate.load("sacrebleu")


def ask(prompt, prompt_template, max_new_tokens=128):
    # 매개변수로 받은 템플릿 사용
    input_text = prompt_template.format(prompt=prompt)
    # ### Instruction: 안녕 ... ### Response: 로 포장
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,  # 창의성 조절
            top_p=0.9,  # 답변의 필터링
            repetition_penalty=1.2, # 같은 말 반복 방지
            do_sample=True, # 매번 조금씩 다른 답변 생성
            eos_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 템플릿의 마지막 줄을 찾아 답변 부분만 분리
    separator = prompt_template.split("\n")[-1].strip()
    if separator in generated_text:
        answer = generated_text.split(separator)[-1].strip()
    else:
        answer = generated_text.strip()
        
    return answer


# 정성적 확인 (직접 질문 던지기)
print("\n--- [정성적 평가: 생성 결과 확인] ---")
test_prompts = [
    "서울역사박물관은 어떤 곳인가요?",
    "점심 메뉴로 매콤한 음식을 추천해줘.",
    "인공지능이란 무엇인지 짧게 설명해봐."
]
for p in test_prompts:
    # 💡 템플릿 인자를 명시적으로 전달합니다.
    response = ask(p, CFG["PROMPT_TEMPLATE"]) 
    print(f"Q: {p}\nA: {response}\n{'-'*30}")



# 정량적 평가 (샘플링하여 ROUGE 점수 확인)
print("\n--- [정량적 평가: ROUGE 스코어 확인 중] ---")

# 10개의 문장...
eval_subset = eval_dataset.select(range(min(10, len(eval_dataset))))
# refs 정답, preds AI의 답
preds, refs = [], []


for sample in eval_subset:
    # 텍스트 디코딩 및 프롬프트 추출
    full_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
    raw_prompt = full_text.split("### Instruction(명령어):")[1].split("### Response(응답):")[0].strip()
  
    
    preds.append(ask(raw_prompt, CFG["PROMPT_TEMPLATE"], max_new_tokens=64))
    
    actual_label = [t for t in sample['labels'] if t != -100]
    refs.append(tokenizer.decode(actual_label, skip_special_tokens=True))


# 메트릭 계산 및 출력
if preds and refs:
    # 1. ROUGE 점수 계산
    rouge_results = rouge.compute(predictions=preds, references=refs)
    
    # 2. BLEU 점수 계산 (sacrebleu는 reference를 리스트의 리스트 형태로 받습니다)
    # sacrebleu는 0~100점 사이로 점수를 줍니다.
    bleu_results = bleu.compute(predictions=preds, references=[[r] for r in refs])
    
    print("\n" + "="*50)
    print("📊 [최종 정량적 평가 결과]")
    print(f"✅ ROUGE-L Score: {rouge_results['rougeL']:.4f} (내용 유사도)")
    print(f"✅ BLEU Score   : {bleu_results['score']:.2f} (문장 유창성)")
    print("="*50)

🚀 원본 모델(Base Model) 로드 중...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 LoRA 어댑터 부착 중...

--- [정성적 평가: 생성 결과 확인] ---
Q: 서울역사박물관은 어떤 곳인가요?
A: '서울역사박물관입니다.
------------------------------
Q: 점심 메뉴로 매콤한 음식을 추천해줘.
A: '저는 인공지능 언어모델로써, 일반적으로 어떤 음식이 점심에 적합한지 알 수 없습니다. 하지만, 일반적으로는 매운맛이 나면서 부드러운 식감의 음식들이 좋습니다.
------------------------------
Q: 인공지능이란 무엇인지 짧게 설명해봐.
A: '인공 지능은 인간의 사고와 판단을 모방하는 컴퓨터 프로그램으로, 인공지능의 한 분야입니다.\n\n인간이 자연이나 사물에서 얻은 정보를 바탕으로 지식을 축적하고 이를 토대로 새로운 가치를 창출하여 보다 나은 삶을 영위할 수 있도록 돕습니다. 또한, 인간 고유의 능력 중 하나인 창의성과 소통 능력을 향상시키는 데에도 도움을 줍니다.\n\n그러나 이러한 인공적인 AI는 인간이 만들어 낸 것이 아니라 자연의 일부이며, 자연 속에서 인간과 상호작용하며 발전해왔기 때문에 인지와 학습, 추론 등의 능력은 자연적으로 형성되는 것입니다. 따라서 인공 지능 역시 인간의 기술과 노하우가 투입되어 만들어진 기술이라고 할 수 있습니다.
------------------------------

--- [정량적 평가: ROUGE 스코어 확인 중] ---

📊 [최종 정량적 평가 결과]
✅ ROUGE-L Score: 0.0286 (내용 유사도)
✅ BLEU Score   : 1.38 (문장 유창성)


# [Phase 2] Reward Model (RM) 학습 코드

In [12]:
class TrinityRM(RewardModel):
    def __init__(self, pretrained=None, lora_rank=8, tokenizer=None):
        full_model = AutoModelForCausalLM.from_pretrained(
            pretrained, 
            dtype=torch.bfloat16, 
            device_map="auto"
        )
            
        value_head = nn.Linear(full_model.config.n_embd, 1).to(torch.bfloat16)
            
        super().__init__(full_model.transformer, value_head, lora_rank)
    
    def save_pretrained(self, save_path):
        import os
        if not os.path.exists(save_path):
            os.makedirs(save_path)
            
        # 1. 내부의 Transformer(LoRA가 적용된 모델) 저장
        self.model.save_pretrained(save_path)
        
        # 2. 추가적인 모델 설정이나 레이어가 있다면 저장 로직을 더 넣을 수 있습니다.
        # (TrinityRM은 부모인 RewardModel 구조를 따르므로 self.model 저장으로 충분합니다.)
        print(f"✅ 모델이 {save_path}에 성공적으로 저장되었습니다.")
          

with NaiveStrategy().model_init_context():
    rm_model = TrinityRM(
        pretrained=CFG["MODEL_NAME"], 
        lora_rank=CFG["LORA_RANK"], 
        tokenizer=tokenizer
    ).to(CFG["DEVICE"])


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import json
from datasets import load_dataset

rm_data_path = f"{CFG['KOCHATGPT_PATH']}/kochatgpt_2_RM.jsonl"

with open(rm_data_path, "r", encoding='utf-8-sig') as f:
    raw_rm_data = json.load(f)

# 🚀 데이터 증량 및 정제 함수 정의
def build_augmented_rm_data(raw_data):
    pairwise_data = []
    
    # 전략 1: 최고의 성적순과 최악의 성적순을 명확히 대조 (Ranking 차이 극대화)
    for item in raw_data:
        prompt = item['prompt']
        ranking = item['ranking']
        winner_idx = ranking.index(0)               # 1등 답변 (Chosen)
        worst_idx = ranking.index(max(ranking))     # 꼴찌 답변 (Rejected)
        
        pairwise_data.append({
            'prompt': prompt,
            'chosen': item[f'completion_{winner_idx}'],
            'rejected': item[f'completion_{worst_idx}']
        })

    # KorQuAD 2.0 스타일 고품질 데이터 추가 (데이터 증량)
    # 실제 정답 데이터를 chosen으로, 모델이 헛소리한 것을 rejected로 시뮬레이션
    try:
        print("💡 외부 데이터셋(KorQuAD 기반) 로드 중...")
        ext_dataset = load_dataset("squad_kor_v1", split="train[:500]") # 500개 추가
        for doc in ext_dataset:
            pairwise_data.append({
                'prompt': doc['question'],
                'chosen': doc['answers']['text'][0],
                'rejected': "질문에 대한 정확한 답을 찾을 수 없습니다." # 저품질 예시
            })
        print(f"✅ 데이터 증량 완료 (총 {len(pairwise_data)} 개 페어)")
    except Exception as e:
        print(f"⚠️ 외부 데이터 로드 실패({e}), 기존 데이터로만 진행합니다.")

    return pairwise_data


pairwise_data = build_augmented_rm_data(raw_rm_data)
train_rm_dataset = RewardDataset(pairwise_data, tokenizer, max_length=512)


💡 외부 데이터셋(KorQuAD 기반) 로드 중...
✅ 데이터 증량 완료 (총 10720 개 페어)


  0%|          | 0/10720 [00:00<?, ?it/s]

In [ ]:
# 학습 데이터 중 일부를 떼어서 모델이 채점을 얼마나 잘하고 있는지 확인합니다.
random.seed(42)
random.shuffle(pairwise_data)
eval_size = 100
train_data = pairwise_data[:-eval_size]
eval_data = pairwise_data[-eval_size:]

train_rm_dataset = RewardDataset(train_data, tokenizer, 512)
eval_rm_dataset = RewardDataset(eval_data, tokenizer, 512)

# 옵티마이저 설정 (LoRA 레이어만 학습할 수 있도록 파라미터 전달)
optimizer = torch.optim.AdamW(rm_model.parameters(), lr=5e-5, weight_decay=0.01)

# 3. RM Trainer 초기화
trainer = RewardModelTrainer(
    model=rm_model,
    strategy=NaiveStrategy(),
    optim=optimizer,
    train_dataset=train_rm_dataset,
    eval_dataset=eval_rm_dataset,
    batch_size=1,            # M4 메모리 안전성 확보
    max_epochs=1,            # 1에폭만으로도 기본적인 채점 감각을 익힙니다
)

# 4. RM 학습 시작 🚀
print("🚀 [RM Phase] Trinity 1.2B 채점 모델 학습을 시작합니다...")
trainer.fit(use_lora=CFG["LORA_RANK"]) 




  0%|          | 0/10620 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

🚀 [RM Phase] Trinity 1.2B 채점 모델 학습을 시작합니다...


Train epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Train step of epoch 0:   0%|          | 0/10620 [00:00<?, ?it/s]

AttributeError: 'TrinityRM' object has no attribute 'save_pretrained'

In [14]:
rm_model.save_pretrained(CFG["RM_MODEL_OUTPUT_PATH"])
tokenizer.save_pretrained(CFG["RM_MODEL_OUTPUT_PATH"])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 모델이 models/RM에 성공적으로 저장되었습니다.


('models/RM/tokenizer_config.json', 'models/RM/tokenizer.json')

In [15]:
def get_reward(prompt, answer):
    input_text = f"### Instruction(명령어):\n{prompt}\n\n### Response(응답):{answer}"
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(CFG["DEVICE"])
    # 모델의 forward pass를 통해 점수(스칼라 값)를 얻습니다.
    reward = rm_model(input_ids)
    return reward.item()


p = "대한민국의 수도는 어디인가요?"
a_good = "대한민국의 수도는 서울입니다."
a_bad = "수도는 물이 나오는 곳입니다."
print(f"Good Answer Score: {get_reward(p, a_good):.4f}")
print(f"Bad Answer Score: {get_reward(p, a_bad):.4f}")

Good Answer Score: -0.1484
Bad Answer Score: -0.1846


# [Phase 3] Proximal Policy Optimization (PPO) 학습 로직

In [6]:
with NaiveStrategy().model_init_context():
    # 🚀 CFG["MODEL_OUTPUT_PATH"]: SFT 모델이 저장된 폴더 (Actor의 모태)
    # 🚀 CFG["RM_MODEL_OUTPUT_PATH"]: RM 모델이 저장된 폴더 (Critic의 모태)
    
    # 1. Actor: 강화학습을 통해 직접 말을 배울 모델
    actor = GPTActor(
        pretrained=CFG["MODEL_OUTPUT_PATH"], 
        lora_rank=CFG["LORA_RANK"]
    ).to(CFG["DEVICE"]).float()
    
    # 2. Critic: 답변의 가치를 스스로 판단해보는 보조 모델
    critic = GPTCritic(
        pretrained=CFG["RM_MODEL_OUTPUT_PATH"], 
        lora_rank=CFG["LORA_RANK"]
    ).to(CFG["DEVICE"]).float()
    
    # 3. Initial Model (Reference): 말투가 너무 변하지 않게 잡아주는 기준점
    # SFT 모델을 다시 로드한 후, 모든 가중치를 얼립니다(Freeze).
    initial_model = GPTActor(
        pretrained=CFG["MODEL_OUTPUT_PATH"], 
        lora_rank=CFG["LORA_RANK"]
    ).to(CFG["DEVICE"]).float()
    for param in initial_model.parameters():
        param.requires_grad = False
    initial_model.model.eval()
    
    # 4. Reward Model: 실제 보상 점수를 주는 엄격한 채점자
    # RM 가중치를 불러온 후, 모든 가중치를 얼립니다(Freeze).
    _temp_rm = GPTCritic(
        pretrained=CFG["RM_MODEL_OUTPUT_PATH"], 
        lora_rank=CFG["LORA_RANK"]
    ).to(CFG["DEVICE"]).float()
    
    reward_model = RewardModel(_temp_rm.model, _temp_rm.value_head).to(CFG["DEVICE"])
    for param in reward_model.parameters():
        param.requires_grad = False
    reward_model.eval()

# 5. 메모리 세이빙: 학습 대상 모델(Actor, Critic)에만 그래디언트 체크포인팅 활성화
actor.model.gradient_checkpointing_enable()
critic.model.gradient_checkpointing_enable()

print(f"✅ SFT({CFG['MODEL_OUTPUT_PATH']})와 RM({CFG['RM_MODEL_OUTPUT_PATH']})로부터 PPO 모델 로드 완료!")


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights: 0it [00:00, ?it/s]

GPT2LMHeadModel LOAD REPORT from: models/SFT
Key                                                                       | Status     | 
--------------------------------------------------------------------------+------------+-
base_model.model.transformer.h.{0...23}.attn.c_attn.lora_A.default.weight | UNEXPECTED | 
base_model.model.transformer.h.{0...23}.attn.c_attn.lora_B.default.weight | UNEXPECTED | 
transformer.h.{0...23}.attn.c_attn.lora_A.default.weight                  | MISSING    | 
transformer.h.{0...23}.attn.c_attn.lora_B.default.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

...

GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights: 0it [00:00, ?it/s]

GPT2LMHeadModel LOAD REPORT from: models/SFT
Key                                                                       | Status     | 
--------------------------------------------------------------------------+------------+-
base_model.model.transformer.h.{0...23}.attn.c_attn.lora_A.default.weight | UNEXPECTED | 
base_model.model.transformer.h.{0...23}.attn.c_attn.lora_B.default.weight | UNEXPECTED | 
transformer.h.{0...23}.attn.c_attn.lora_A.default.weight                  | MISSING    | 
transformer.h.{0...23}.attn.c_attn.lora_B.default.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

✅ SFT(models/SFT)와 RM(models/RM)로부터 PPO 모델 로드 완료!


In [7]:
# PPO용 데이터 로드 (프롬프트만 사용)
ppo_data_path = f"{CFG['KOCHATGPT_PATH']}/kochatgpt_3_PPO.jsonl"
with open(ppo_data_path, "r", encoding='utf-8-sig') as f:
    raw_ppo_data = json.load(f)
    
# 프롬프트 리스트 추출
list_prompt = [tmp['prompt'] for tmp in raw_ppo_data]


tokenizer = PreTrainedTokenizerFast.from_pretrained(
    CFG["MODEL_OUTPUT_PATH"],
    bos_token='</s>', 
    eos_token='</s>', 
    unk_token='<unk>', 
    pad_token='<pad>', 
    mask_token='<mask>', 
    padding_side="right",
    model_max_length=512
)



# 토크나이징 함수 정의 (PPO 학습 중 실시간 토크나이징을 위해 필요)
def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.to(CFG["DEVICE"]) for k, v in batch.items()}

In [8]:
actor_optim = torch.optim.AdamW(actor.parameters(), lr=1e-6)
critic_optim = torch.optim.AdamW(critic.parameters(), lr=1e-6)


(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model
)


In [ ]:
# 2. PPO 트레이너 설정
# 이제 여기서 전달되는 actor, critic 등은 이미 .prepare()를 거쳐 
# 하드웨어 및 학습 환경에 최적화된 상태의 객체들입니다.
trainer = PPOTrainer(
    strategy=strategy,       # 새로 생성하기보다 변수화된 strategy 사용 권장
    actor=actor,             # 이미 prepare된 actor
    critic=critic,           # 이미 prepare된 critic
    reward_model=reward_model,
    initial_model=initial_model,
    actor_optim=actor_optim, # 이미 prepare된 optimizer
    critic_optim=critic_optim,# 이미 prepare된 optimizer
    tokenizer=tokenize_fn,
    max_epochs=1,
    train_batch_size=1,      
    experience_batch_size=4, 
    max_length=64,           
    do_sample=True,
    temperature=1.0,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id
)


# 2. PPO 학습 시작!
print("🚀 [PPO Phase] 최종 강화학습을 시작합니다...")
# 에피소드 수를 적절히 조절하여 전체 학습 시간을 관리하세요.
trainer.fit(list_prompt, num_episodes=10, max_timesteps=3, update_timesteps=3)

# 3. 최종 업그레이드된 챗봇 저장
# 이 모델이 마침내 SFT + RM + PPO를 모두 마친 'Custom ChatGPT'입니다.
actor.model.save_pretrained(CFG["PPO_MODEL_OUTPUT_PATH"])
tokenizer.save_pretrained(CFG["PPO_MODEL_OUTPUT_PATH"])
print(f"✅ 축하합니다! 최종 모델이 {CFG["PPO_MODEL_OUTPUT_PATH"]}에 저장되었습니다.")


🚀 [PPO Phase] 최종 강화학습을 시작합니다...


Episode [1/10]:   0%|          | 0/3 [00:00<?, ?it/s]

: 